# vbOCP — Test 1: pipeline completa

Notebook orchestratore bash-style: le celle di codice lanciano solo comandi verso gli script della repo — nessuna logica del modello qui dentro. Stesso pattern per Colab/cluster: cambia solo dove lo lanci.

**Contenuto:**
1. Setup (path della repo)
2. FOM — solve singolo + plot, generazione snapshot
3. ROM (POD) — training + plot decadimento autovalori
4. GNN *(da aggiungere)*
5. Autoencoder *(da aggiungere)*

## 1. Setup

In [ ]:
import os

# si sposta nella root della repo (dove c'e' src/), indipendentemente da dove parte il kernel
while not os.path.isdir('src') and os.getcwd() != os.path.dirname(os.getcwd()):
    os.chdir('..')
assert os.path.isdir('src'), "src/ non trovata: verifica dove e' montata la repo vbOCP"

REPO_ROOT      = os.getcwd()
CONFIG_PATH    = os.path.join('configs', 'test1.yaml')
DATA_DIR       = os.path.join('data')
SNAPSHOTS_DIR  = os.path.join('data', 'snapshots')
OUTPUT_DIR     = os.path.join('notebooks', 'output')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SNAPSHOTS_DIR, exist_ok=True)

print('REPO_ROOT    :', REPO_ROOT)
print('CONFIG_PATH  :', CONFIG_PATH)
print('SNAPSHOTS_DIR:', SNAPSHOTS_DIR)

## 2. FOM

### 2.1 Solve singolo + plot

Risolve il FOM per una terna di parametri e salva un confronto visivo di stato `y` e aggiunto `p`. Modifica `--mu1`/`--mu2`/`--mu_u` per esplorare altri punti dello spazio parametrico.

In [ ]:
FOM_PLOT_PATH = os.path.join(OUTPUT_DIR, 'fom_solution.png')

!python -m src.full_order.run_single_solve \
    --config {CONFIG_PATH} \
    --mu1 12 \
    --mu2 2.5 \
    --mu_u 0.99 \
    --plot --output {FOM_PLOT_PATH}

from IPython.display import Image
Image(FOM_PLOT_PATH)

### 2.2 Generazione snapshot

Campiona `mu1`, `mu2`, `mu_u` uniformemente sui range definiti nel config e risolve il FOM per ciascun campione. L'assembly indipendente da mu viene fatto una sola volta; il ciclo aggiorna solo la matrice di controllo e il solve.

Output: un unico file `.npz` con `mu1`, `mu2`, `mu_u`, e le matrici `Y`, `P`, `U` (una colonna per campione).

In [ ]:
N_SAMPLES = 300
SNAPSHOTS_PATH = os.path.join(SNAPSHOTS_DIR, f'test1_{N_SAMPLES}.npz')

!python -m src.full_order.generate_snapshots \
    --config {CONFIG_PATH} \
    --n-samples {N_SAMPLES} \
    --output {SNAPSHOTS_PATH}

In [ ]:
import numpy as np

data = np.load(SNAPSHOTS_PATH)
print('Chiavi salvate:', list(data.keys()))
print('Y shape:', data['Y'].shape)

## 3. ROM (POD)

Costruisce la base POD per stato e aggiunto a partire dagli snapshot generati sopra. Prodotto scalare e numero di modi configurabili qui sotto; `--plot` e' opzionale.

In [ ]:
INNER_PRODUCT = 'seminorm'  # 'seminorm' (solo A_diff) o 'full' (A_diff + M_full)
N_MODES = 30
POD_PATH = os.path.join(SNAPSHOTS_DIR, 'test1_pod.npz')
POD_PLOT_PATH = os.path.join(OUTPUT_DIR, 'pod_eigenvalue_decay.png')

!python -m src.rom.train_pod \
    --config {CONFIG_PATH} \
    --snapshots {SNAPSHOTS_PATH} \
    --inner-product {INNER_PRODUCT} \
    --n-modes {N_MODES} \
    --output {POD_PATH} \
    --plot --plot-output {POD_PLOT_PATH}

from IPython.display import Image
Image(POD_PLOT_PATH)

## 4. GNN

*(da aggiungere)*

## 5. Autoencoder

*(da aggiungere)*